In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Celda 1: Imports y Configuración de Path
import sys
import os
# Aseguramos que Python vea la carpeta 'src'
sys.path.append(os.path.abspath(os.path.join('..')))

# Importamos CADA clase de SU fichero
from src.ingestor import IngestorDatos      # Mod 1
from src.preparador import PreparadorDatos  # Mod 2
from src.analizador import Analizador       # Mod 3
from src.modelos import EvaluadorRiesgo     # Mod 4
from src.presentacion import Visualizador   # Mod 5
from src.trazabilidad import GestorLogs     # Mod 6

# Celda 2: Ejecución del Pipeline
def main():
    # 0. Iniciar Logs
    logs = GestorLogs()
    logs.registrar("ORQUESTADOR", "Iniciando proceso de integración...")

    # 1. Ingesta
    try:
        ingestor = IngestorDatos() # Usa tu lógica de rutas automática
        datasets = ingestor.leer_datos()
        logs.registrar("MODULO_1", f"Datos cargados: {list(datasets.keys())}")
    except Exception as e:
        logs.registrar("ERROR_M1", str(e))
        return

    # 2. Preparación
    try:
        preparador = PreparadorDatos(datasets)
        df_master = preparador.ejecutar_preparacion()
        logs.registrar("MODULO_2", f"Tabla maestra creada con {len(df_master)} filas")
    except Exception as e:
        logs.registrar("ERROR_M2", str(e))
        return

    # 3. Análisis
    analista = Analizador()
    stats = analista.calcular_estadisticas_basicas(df_master)
    logs.registrar("MODULO_3", f"Estadísticas calculadas: {stats}")

    # 4. Modelado (Riesgo)
    evaluador = EvaluadorRiesgo()
    df_riesgo = evaluador.ejecutar_evaluacion(df_master)
    logs.registrar("MODULO_4", "Riesgos asignados correctamente")

    # 5. Presentación (Consola y Fichero)
    vista = Visualizador(output_dir='../output') # Apuntamos a la carpeta output externa
    vista.mostrar_en_consola(df_riesgo, stats)
    vista.exportar_csv(df_riesgo)
    logs.registrar("MODULO_5", "Reporte generado en carpeta /output")

    logs.registrar("ORQUESTADOR", "Proceso finalizado con éxito.")

# Ejecutar
if __name__ == "__main__":
    main()

--- Inicio de Sesión: 2026-02-12 15:49:31 ---
[15:49:31] [ORQUESTADOR] Iniciando proceso de integración...
Cargando inscripciones desde LSE_Inscrip_Baja_Recibido.csv...
Cargando notas_bimestre desde LSE_Notas_Estadistica_Bimestre.csv...
Cargando actual desde LSE_Notas_Inscrip_Baja_Actual.xlsx...
[15:49:32] [MODULO_1] Datos cargados: ['inscripciones', 'notas_bimestre', 'actual']
Tabla Maestra generada con 772 registros.
[15:49:32] [MODULO_2] Tabla maestra creada con 772 filas
[15:49:32] [MODULO_3] Estadísticas calculadas: {'columna_analizada': 'nota_b1', 'media_aritmetica': np.float64(8.64), 'total_alumnos': 772}
[15:49:32] [MODULO_4] Riesgos asignados correctamente

   REPORTE DE SEGUIMIENTO ACADÉMICO
Alumnos procesados: 772
Nota media global (nota_b1): 8.64
----------------------------------------
Muestra de Riesgos Asignados:
     n_siu estado_actual nivel_riesgo
0  2653003           NaN        MEDIO
1  2653003           NaN        MEDIO
2  2653003      Recibido         BAJO
3  26820